In [ ]:
#Installing the pre-requirement and active the Spark Environment
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.4.1/spark-3.4.1-bin-hadoop3.tgz
!tar xf spark-3.4.1-bin-hadoop3.tgz
!pip install -q findspark

In [ ]:
#Environment Variables of Java and Spark
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.4.1-bin-hadoop3"

In [ ]:
# Install the GraphFrames library appropriate for the Spark version
!pip install graphframes

In [ ]:
#The necessay libraries
import findspark
findspark.init()

import time
import random
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from graphframes import GraphFrame

In [ ]:
#2. Create a Spark session with the GraphFrames graphics package
spark = SparkSession.builder \
    .appName("SparkGraphBenchmarks") \
    .config("spark.jars.packages", "graphframes:graphframes:0.8.3-spark3.4-s_2.12") \
    .getOrCreate()
sc = spark.sparkContext
sc.setCheckpointDir("/tmp/checkpoints")

In [ ]:
# 3.Inputs
user_counts = [100, 500, 1000]
trip_counts = [100, 500, 1000]
events_options = [0, 2, 5, 10]
num_stations = 50

In [ ]:
# 4.Simulating data and making Graph in memory spark
def generate_spark_graph(n_users, n_trips, n_events):
    # Creating random values for stations
    stations_list = [(i, f"station_{i}") for i in range(1, num_stations + 1)]
    vertices_df = spark.createDataFrame(stations_list, ["id", "name"])

    # Random edges among the graphs
    edges_list = []
    for j in range(1, n_trips + 1):
        src_station = random.randint(1, num_stations)
        dst_station = random.randint(1, num_stations)
        edges_list.append((src_station, dst_station, f"T{j}"))

    edges_df = spark.createDataFrame(edges_list, ["src", "dst", "trip_id"])

    # making building
    g = GraphFrame(vertices_df, edges_df)
    return g

In [ ]:
# 5.Functions for Queries

# 1st Query---->
def run_pagerank_query(g):
    start_q = time.time()
    pr_result = g.pageRank(resetProbability=0.15, maxIter=5)
    # find 3 most important stations
    top_3 = pr_result.vertices.orderBy(col("pagerank").desc()).limit(3)
    top_3.toPandas()
    # end time
    end_q = time.time()
    return round(end_q - start_q, 4)

#2nd Query
def run_connected_components_query(g):
    start_q = time.time()
    cc_result = g.connectedComponents()
    cc_result.limit(5).toPandas() # load data for timer
    end_q = time.time()
    return round(end_q - start_q, 4)


# 6.main loop
round_num = 1
spark_benchmark_results = []

for n_users in user_counts:
    for n_trips in trip_counts:
        for events in events_options:
            print(f"\n[Spark] Execution Round #:{round_num}/36")
            print(f"Config: Users={n_users}, Trips={n_trips}, Events/Trip={events}")
            print("------------------------------------------------------------")

            start_time = time.time()
            g = generate_spark_graph(n_users, n_trips, events)
            end_time = time.time()
            generation_duration = round(end_time - start_time, 4)

            print(f"Graph loaded into memory in {generation_duration} seconds.")

            # Run algorithm and record time
            q1_duration = run_pagerank_query(g)
            q2_duration = run_connected_components_query(g)

            print(f"PageRank: {q1_duration}s | ConnectedComponents: {q2_duration}s")

            #Save in list
            spark_benchmark_results.append({
                "Round": round_num,
                "Num_Users": n_users,
                "Num_Trips": n_trips,
                "Events_Per_Trip": events,
                "Graph_Load_Time_Sec": generation_duration,
                "PageRank_Query1_Sec": q1_duration,
                "ConnectedComponents_Query2_Sec": q2_duration
            })
            round_num += 1


[Spark] Execution Round #:1/36
Config: Users=100, Trips=100, Events/Trip=0
------------------------------------------------------------


/content/spark-3.4.1-bin-hadoop3/python/pyspark/sql/dataframe.py:169: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(


Graph loaded into memory in 4.9956 seconds.


/content/spark-3.4.1-bin-hadoop3/python/pyspark/sql/dataframe.py:148: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")
/content/spark-3.4.1-bin-hadoop3/python/pyspark/sql/dataframe.py:148: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


PageRank: 18.0139s | ConnectedComponents: 127.2944s

[Spark] Execution Round #:2/36
Config: Users=100, Trips=100, Events/Trip=2
------------------------------------------------------------
Graph loaded into memory in 0.1171 seconds.


...

PageRank: 3.2442s | ConnectedComponents: 62.9196s

[Spark] Execution Round #:35/36
Config: Users=1000, Trips=1000, Events/Trip=5
------------------------------------------------------------
Graph loaded into memory in 0.0997 seconds.


/content/spark-3.4.1-bin-hadoop3/python/pyspark/sql/dataframe.py:169: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(
/content/spark-3.4.1-bin-hadoop3/python/pyspark/sql/dataframe.py:148: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")
/content/spark-3.4.1-bin-hadoop3/python/pyspark/sql/dataframe.py:148: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


PageRank: 6.6294s | ConnectedComponents: 73.5639s

[Spark] Execution Round #:36/36
Config: Users=1000, Trips=1000, Events/Trip=10
------------------------------------------------------------
Graph loaded into memory in 0.0829 seconds.


/content/spark-3.4.1-bin-hadoop3/python/pyspark/sql/dataframe.py:169: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(
/content/spark-3.4.1-bin-hadoop3/python/pyspark/sql/dataframe.py:148: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")
/content/spark-3.4.1-bin-hadoop3/python/pyspark/sql/dataframe.py:148: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


PageRank: 7.5366s | ConnectedComponents: 187.4486s


In [ ]:
# 7.Save in EXCLE
import pandas as pd
df_spark = pd.DataFrame(spark_benchmark_results)
df_spark.to_csv("spark_benchmark_results.csv", index=False)
print("\n[SUCCESS] Spark benchmarking finished! 'spark_benchmark_results.csv' saved. 📊")


[SUCCESS] Spark benchmarking finished! 'spark_benchmark_results.csv' saved. 📊
